In [3]:
from gsloc.inference.test import TestConfig, Test
from pathlib import Path
from gsloc.models import opr_graph_extention as network 
import torch
from torchvision.transforms import functional as F
from mmpr.models import MegaLoc
from gsloc.datasets import ThreeRScan

from torchvision import transforms as T
from gsloc.utils.visual import plot_metrics_from_parquet, plot_metrics_from_experiment_dir

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


2026-05-07 11:52:09.613 | WARNING  | opr.optional_deps:warn_once:115 - MinkowskiEngine is not available. sparse convolutions will be disabled. See the documentation for installation instructions


In [2]:
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
test_dir = Path("/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc")
index_path = "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256_graph_index"
rerank_index_path = "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/megaloc_index"
query_cache_path = "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256_graph_query_cache"
rerank_query_cache_path = "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/megaloc_query_cache"

graph_path = "SceneGraphs_Makarov_FULL_TEST_pt"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

bench_report_dir = test_dir / "bench_reports" / "pose-near-sim"
frames_path = test_dir / "bench_reports" / "frames.npz"

seq_filter_kwargs_list = [{
    "seq_similarity_filter_mode": "none",
},
{
    "seq_similarity_filter_mode": "pose",
    "seq_similarity_trans_tol_m": 0.5,
    "seq_similarity_rot_tol_deg": 15
},{
    "seq_similarity_filter_mode": "pose",
    "seq_similarity_trans_tol_m": 1,
    "seq_similarity_rot_tol_deg": 30
},
]
models = [graph_model1]
rerank_models = [megaLoc]

graph_path_list = [graph_path, "SceneGraphs_real_classes_pt_compact"]
similarity_kwargs_list = [
    {
        "mode": "room",
    },
    {
        "mode": "pose",
        "trans_tol_m": 3,
        "rot_tol_deg": 180
    },
    {
        "mode": "pose",
        "trans_tol_m": 2,
        "rot_tol_deg": 90
    },
]


image_transform_fn = T.Compose([
    T.ToTensor(),
    T.Lambda(lambda x: F.rotate(x, angle=-90)),  # 90° clockwise
    T.Normalize(mean=[0.44420420130352495, 0.41322746532289134, 0.3678658064565412], std=[0.24352604373543688, 0.24045797651069503, 0.24250136992133814]),
    T.Resize([322, 322], antialias=True)
])

cfg = TestConfig(
    dataset_path=dataset_path,
    test_path=test_dir,
    index_path=index_path,
    rerank_index_path=rerank_index_path,
    query_cache_path=query_cache_path,
    rerank_query_cache_path=rerank_query_cache_path,
    bench_report_path=bench_report_dir,
    graph_path=graph_path,   
    dataset_class=ThreeRScan,
    filter_kwargs={"similarity_filter_mode": "none", "similarity_trans_tol_m": 2, "similarity_rot_tol_deg": 90},
    seq_filter_kwargs={"seq_similarity_filter_mode": "none", "seq_similarity_trans_tol_m": 2, "seq_similarity_rot_tol_deg": 90},
    scene_list_path=scene_list_path,
    room_json_path=room_json_path,
    edge_normalizer_path=edge_normalizer_path,
    image_transform_fn=image_transform_fn,
    graph_feat_dim=4,
    graph_edge_attr_dim=10,
    graph_rotate=True,
    device=device,
    batch_size=16,
    num_workers=4,
    model=graph_model1,
    rerank_model=megaLoc,
    rerank_k=500,
    per_frame_k_used=25,
    final_k=25,
    seq_lengths=[1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 35],
    recall_at_k=[1, 5, 10, 25],
    similarity_kwargs=similarity_kwargs_list[2],
    std_mode="global",
    scene_df_field="scene",
    pose_df_field="pose",
    frames_path=frames_path
)

NameError: name 'Path' is not defined

In [5]:
similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]

In [6]:
# cfg.seq_filter_kwargs = seq_filter_kwargs_list[1]
# cfg.bench_report_path = cfg.test_path / filter_names[1] / similarity_names[0]
# cfg.frames_path = cfg.test_path / "frames.npz"
# test = Test(cfg)
# test.run()

In [5]:
plot_metrics_from_parquet(
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/rerank_k_50/nearfilter_seq_report/pose-far-sim/summaryresults.parquet",
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/rerank_k_100/nearfilter_seq_report/pose-far-sim/summaryresults.parquet",
    metrics=("recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"))

{'recall_at_1': Figure({
     'data': [{'error_y': {'array': {'bdata': ('Ut/YJZCnqj+RVSEjt4KoP9OZSeSL2a' ... '4H1ag/alaa1ADupj80ntJ6dNOoPw=='),
                                     'dtype': 'f8'}},
               'hovertemplate': 'w=%{x}<br>recall_at_1=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('wkWapQs04D9x2SgLvCHjPy0riLsjH+' ... 'tcNeQ/xu13XHJB5D9zA5G9WFzkPw=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend

In [5]:
plot_metrics_from_parquet(
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/rerank_k_500/nearfilter_seq_report/pose-far-sim/summaryresults.parquet",
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/Megaloc/nearfilter_seq_report/pose-far-sim/summaryresults.parquet",
    metrics=("recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"))

{'recall_at_1': Figure({
     'data': [{'error_y': {'array': {'bdata': ('ug8YavLgpj9UBrzqmMKnP7AcnNMfoa' ... 'z8M6c/nDfnGo9xqT8c1Gc0d5mnPw=='),
                                     'dtype': 'f8'}},
               'hovertemplate': 'w=%{x}<br>recall_at_1=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('hzpME5AW5D+DuDvyQS/lP0dwAGyjsu' ... 'eZX+U/Iws2/Bo15T9DmuL4JzPlPw=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend

In [ ]:
plot_metrics_from_parquet(
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/nearfilter_seq_report/pose-far-sim/summaryresults.parquet",
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/base_seq_report/pose-far-sim/summaryresults.parquet",
    metrics=("recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"))

NameError: name 'plot_metrics_from_parquet' is not defined

In [ ]:
plot_metrics_from_parquet(
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/farfilter_seq_report/room-sim/summaryresults.parquet",
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/nearfilter_seq_report/room-sim/summaryresults.parquet",
    metrics=("recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"))

{'recall_at_1': Figure({
     'data': [{'error_y': {'array': {'bdata': ('cUbi3p/Eoj/UO53XYuWgP/INdHIup5' ... 'xPTZ0/PwxUm4yonj+pJaQrp/WbPw=='),
                                     'dtype': 'f8'}},
               'hovertemplate': 'w=%{x}<br>recall_at_1=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('iJi4kHWW6j/oLp4oklfsPxHVs7vNJe' ... 'ZuFe0/JaFBYc0O7T93LjKF30HtPw=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend

In [ ]:
plot_metrics_from_parquet(
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/farfilter_seq_report/pose-far-sim/summaryresults.parquet",
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/nearfilter_seq_report/pose-far-sim/summaryresults.parquet",
    metrics=("recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"))

{'recall_at_1': Figure({
     'data': [{'error_y': {'array': {'bdata': ('TYtABNdkpz858liCKbOnP3a/Xpwxf6' ... 'jbBKk/wmLbnVjLqT/qGvDYbE2oPw=='),
                                     'dtype': 'f8'}},
               'hovertemplate': 'w=%{x}<br>recall_at_1=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('dNnrHcj+4z/i+hM4MVjlP8Aq3yp95u' ... '3pueQ/K3GbxRS75D8XLKk6reHkPw=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend

In [ ]:
plot_metrics_from_parquet(
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/farfilter_seq_report/pose-near-sim/summaryresults.parquet",
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/nearfilter_seq_report/pose-near-sim/summaryresults.parquet",
    metrics=("recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"))

{'recall_at_1': Figure({
     'data': [{'error_y': {'array': {'bdata': ('86+dfOQCqT9iDaSb7b2pPzV4+a9hEq' ... 'G76qc/ZkEtHyFupj+sqXd+mLanPw=='),
                                     'dtype': 'f8'}},
               'hovertemplate': 'w=%{x}<br>recall_at_1=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('P6hWG0x11D/sEX3OXyjWP7fSsJazf9' ... 'ubq9I/rKj/q/Kt0j83Z0jljEfTPw=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend